# Análise descritiva e estatística da base coletada (etapa de transformação)

Este notebook é **independente do scraper**: ele só lê as tabelas que a coleta gravou
(`rounds`, `mapas` e `partidas`) e produz a base pronta para a modelagem.

Ele faz quatro coisas, nesta ordem:

1. **Inventário da base coletada:** quantos eventos, partidas, mapas e rounds existem antes de qualquer limpeza.
2. **Escopo e limpeza:** tira as partidas fora do escopo do estudo (**showmatches, open qualifiers e séries Bo1**), aplica regras explícitas de validade e mostra o **funil** do total coletado, passando pelos excluídos **fora do escopo** e pelos dados **inválidos** (por regra), até os dados **válidos**.
3. **Análise descritiva** da base válida e **criação das variáveis processadas** (`win_count`, `loss_count`, `score_diff`, sequência de vitórias etc.), todas calculadas *antes* do round, sem vazamento.
4. **Análise estatística** ligada às hipóteses do TCC (H1 a H3): efeito do lado, do tipo de compra, da diferença de loadout e do momentum sobre a vitória no round.

**Entradas:** `<PASTA_DADOS>/processed/*.parquet` (coleta completa) ou os CSVs de amostra do repositório (para testar o notebook).
**Saídas:** `<PASTA_DADOS>/transformed/`, com a base válida, a base com as variáveis processadas, o relatório de limpeza e as figuras.

## 0. Configuração

In [ ]:
import os

# De onde ler os dados:
#   "drive"   -> <PASTA_DADOS>/processed/*.parquet (resultado da coleta)
#   "amostra" -> CSVs de amostra publicados no GitHub (2 partidas reais; só para testar o notebook)
FONTE = os.environ.get("ANALISE_FONTE", "drive")
PASTA_DADOS = os.environ.get("ANALISE_DADOS", "/content/drive/MyDrive/TCC/vlr_dados")
URL_AMOSTRA = os.environ.get("ANALISE_URL_AMOSTRA",
                             "https://raw.githubusercontent.com/isa-dib/vlr_scraper/main/data/amostra")

# Escopo do estudo: partidas excluídas por tipo, antes das regras de validade.
# Elas aparecem no funil como "fora do escopo" (não como dados inválidos).
EXCLUIR_SHOWMATCH = True        # partidas de exibição (Showmatch, All-Star, Exhibition)
EXCLUIR_OPEN_QUALIFIER = True   # qualificatórias abertas (Open Qualifier); as fechadas continuam
EXCLUIR_BO1 = True              # séries melhor de 1
FILTRO_EVENTOS = None           # ex.: [2766, 2977, 2976, 2776]; None = todos
FILTRO_DATA_MIN = None          # ex.: "2023-01-01"; None = sem limite

# Política de limpeza: se um round de um mapa é inválido, descartar o MAPA inteiro?
# True é o recomendado para modelos de sequência (LSTM Seq2Seq): um buraco no meio
# do mapa quebra a sequência e as variáveis acumuladas (win_count, streak...).
MAPA_INTEIRO = True

SALVAR = True                # grava as saídas em <PASTA_DADOS>/transformed
N_BOOT = 1000                # reamostragens do bootstrap por mapa (intervalos de confiança)
SEMENTE = 42

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
rng = np.random.default_rng(SEMENTE)

def col(df, nome, padrao=np.nan):
    """Coluna do DataFrame ou uma série constante, se a coluna não existir (ex.: base sem aba Economy)."""
    return df[nome] if nome in df.columns else pd.Series(padrao, index=df.index)

EM_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if FONTE == "drive" and EM_COLAB and PASTA_DADOS.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

if FONTE == "drive":
    PASTA_SAIDA = Path(PASTA_DADOS) / "transformed"
else:
    PASTA_SAIDA = Path("transformed_amostra")
PASTA_FIG = PASTA_SAIDA / "figuras"
if SALVAR:
    PASTA_FIG.mkdir(parents=True, exist_ok=True)
print("fonte:", FONTE, "| saídas:", PASTA_SAIDA.resolve())

## 1. Leitura e inventário da base coletada

In [ ]:
def ler(nome):
    if FONTE == "drive":
        p = Path(PASTA_DADOS) / "processed" / f"{nome}.parquet"
        if not p.exists():
            raise FileNotFoundError(f"{p} não existe. Rode a coleta (colab.coletar) ou use FONTE = 'amostra'.")
        return pd.read_parquet(p)
    return pd.read_csv(f"{URL_AMOSTRA}/{nome}.csv")

rounds_raw = ler("rounds")
mapas_raw = ler("mapas")
partidas_raw = ler("partidas")
for nome, df in [("rounds", rounds_raw), ("mapas", mapas_raw), ("partidas", partidas_raw)]:
    print(f"{nome:9s} {df.shape[0]:7d} linhas x {df.shape[1]} colunas")

In [ ]:
def salvar_fig(fig, nome):
    if SALVAR:
        fig.savefig(PASTA_FIG / f"{nome}.png", bbox_inches="tight")

inventario = pd.DataFrame({
    "eventos": [partidas_raw["event_id"].nunique()],
    "partidas": [partidas_raw["match_id"].nunique()],
    "mapas listados": [len(mapas_raw)],
    "mapas jogados": [int((~mapas_raw["status"].isin(["nao_jogado", "sem_mapas"])).sum())],
    "mapas com rounds": [rounds_raw["game_id"].nunique()],
    "rounds": [len(rounds_raw)],
}, index=["base coletada"])
display(inventario)

print("Status dos mapas listados:")
display(mapas_raw["status"].value_counts().rename_axis("status").to_frame("mapas"))

**Mapas não jogados não são dados inválidos.** São mapas que aparecem no veto, mas que a série
não chegou a disputar (por exemplo, o 3º mapa de uma série 2:0). Eles ficam fora de todas as
contagens a seguir. O ponto de partida da limpeza é o conjunto de **mapas jogados**.

## 2. Escopo do estudo: partidas excluídas por tipo

Antes das regras de validade, o notebook tira as partidas que **não fazem parte do estudo**,
mesmo com dados corretos. No funil, elas aparecem como **fora do escopo**, separadas dos inválidos.

| Motivo | Critério | Por quê |
|---|---|---|
| `showmatch` | a etapa (`event_series`) contém "Showmatch", "Show Match", "All-Star" ou "Exhibition" | partida de exibição, sem disputa competitiva real |
| `open_qualifier` | a etapa ou o nome do evento contém "Open Qualifier" (ou a URL tem `-oq-`) | times amadores e sem continuidade; nível competitivo muito heterogêneo (ex.: as Open Qualifiers do Challengers Brazil 2022, com 134 das 167 partidas do evento) |
| `bo1` | `best_of` = "Bo1" (se o formato não estiver no cabeçalho, uma série com um único mapa listado) | o formato muda o comportamento dos times: não há adaptação entre mapas nem gestão de série |
| `evento_fora_da_lista` / `data_anterior` | filtros opcionais `FILTRO_EVENTOS` e `FILTRO_DATA_MIN` | recorte de período ou de eventos |

Cada partida recebe só o **primeiro** motivo da lista, na ordem da tabela. As qualificatórias
**fechadas** (Closed Qualifier) continuam na base.

In [ ]:
RE_SHOWMATCH = r"show\s*-?\s*match|all[\s-]?star|exhibition"
RE_OPEN_QUALIFIER = r"open\s*qual"

partidas = partidas_raw.copy()
partidas["data"] = pd.to_datetime(partidas["date_utc"], errors="coerce")
serie = col(partidas, "event_series", "").fillna("").astype(str).str.lower()
evento = col(partidas, "event_name", "").fillna("").astype(str).str.lower()
url = col(partidas, "match_url", "").fillna("").astype(str).str.lower()
best_of = col(partidas, "best_of")

partidas["tipo_showmatch"] = serie.str.contains(RE_SHOWMATCH)
partidas["tipo_open_qualifier"] = (serie.str.contains(RE_OPEN_QUALIFIER) | evento.str.contains(RE_OPEN_QUALIFIER)
                                   | url.str.contains(r"-oq-|open-qualifier"))
partidas["bo1_inferido"] = best_of.isna() & col(partidas, "mapas_listados").eq(1)
partidas["tipo_bo1"] = best_of.eq("Bo1") | partidas["bo1_inferido"]

motivo_escopo = pd.Series(pd.NA, index=partidas.index, dtype="object")
regras_escopo = [
    ("showmatch", EXCLUIR_SHOWMATCH, partidas["tipo_showmatch"]),
    ("open_qualifier", EXCLUIR_OPEN_QUALIFIER, partidas["tipo_open_qualifier"]),
    ("bo1", EXCLUIR_BO1, partidas["tipo_bo1"]),
    ("evento_fora_da_lista", FILTRO_EVENTOS is not None,
     ~partidas["event_id"].isin(FILTRO_EVENTOS or [])),
    ("data_anterior", FILTRO_DATA_MIN is not None,
     partidas["data"] < pd.Timestamp(FILTRO_DATA_MIN or "1900-01-01")),
]
for nome, ativo, cond in regras_escopo:
    if ativo:
        motivo_escopo[motivo_escopo.isna() & cond.fillna(False)] = nome
partidas["motivo_escopo"] = motivo_escopo
escopo_por_partida = partidas.drop_duplicates("match_id").set_index("match_id")["motivo_escopo"]

mapas_jogados = mapas_raw[~mapas_raw["status"].isin(["nao_jogado", "sem_mapas"])].copy()
mapas_jogados["motivo_escopo"] = mapas_jogados["match_id"].map(escopo_por_partida).astype("object")
rounds_raw["motivo_escopo"] = rounds_raw["match_id"].map(escopo_por_partida).astype("object")
rounds_raw["no_escopo"] = rounds_raw["motivo_escopo"].isna()

ORDEM_ESCOPO = [n for n, *_ in regras_escopo]
tab_escopo = pd.DataFrame({
    "partidas": partidas.drop_duplicates("match_id")["motivo_escopo"].value_counts(),
    "mapas jogados": mapas_jogados["motivo_escopo"].value_counts(),
    "rounds": rounds_raw["motivo_escopo"].value_counts(),
}).reindex(ORDEM_ESCOPO).dropna(how="all").fillna(0).astype(int)
print("Excluídos por escopo:")
display(tab_escopo)
if partidas["bo1_inferido"].any():
    print(f"atenção: {int(partidas['bo1_inferido'].sum())} partida(s) sem formato no cabeçalho foram "
          "tratadas como Bo1 por terem um único mapa listado")

In [ ]:
# Conferência manual: quais textos de etapa caíram em cada motivo (confira se não há falso positivo)
conf = (partidas.dropna(subset=["motivo_escopo"])
          .groupby(["motivo_escopo", "event_series"], dropna=False).size().rename("partidas").reset_index())
display(conf.sort_values(["motivo_escopo", "partidas"], ascending=[True, False]).head(40))

# Etapas com "qual" que FICARAM na base (devem ser apenas qualificatórias fechadas ou ligas)
restam = partidas[partidas["motivo_escopo"].isna() & serie.str.contains("qual")]
if len(restam):
    print("Etapas com 'qual' mantidas na base:")
    display(restam["event_series"].value_counts().head(20).rename_axis("event_series").to_frame("partidas"))

## 3. Limpeza: regras de validade

As regras são de dois níveis. A coluna `remover` diz se a regra descarta o dado ou só gera um aviso.

**Nível mapa:** o mapa inteiro é descartado.

| Regra | Critério | Por quê |
|---|---|---|
| `sem_replay` | o vlr.gg não tem 2D replay do mapa | sem kills, loadout por jogador nem eventos do round |
| `nao_baixado` / `nao_renderizado` | a página não está no cache ou veio vazia | dado ausente: rodar a coleta de novo |
| `inconsistente_parser` | o parser encontrou divergências (feed × JSON, 10 jogadores) | dado não confiável |
| `economia_invalida` | aba Economy ausente ou divergente do replay | sem `type_buy` / `loadout_vlr` confiáveis |
| `mapa_duplicado` | o mesmo `game_id` aparece mais de uma vez | evita contar o mesmo mapa duas vezes |
| `sequencia_incompleta` | os rounds não formam 1, 2, …, n, ou n diverge do total do mapa | a sequência do mapa está quebrada |
| `placar_inconsistente` | o placar final não fecha (vencedor com 13; na prorrogação, diferença de 2; soma = n) | indica rounds faltando ou lidos errado |
| `rounds_invalidos` | o mapa tem ao menos um round inválido (só com `MAPA_INTEIRO = True`) | preserva sequências completas para o LSTM |

**Nível round:**

| Regra | Critério | Remove? |
|---|---|---|
| `vencedor_ausente` | `winner` fora de {1, 2} | sim |
| `vencedor_diverge` | o vencedor pelo placar ≠ o vencedor pela faixa de rounds | sim |
| `jogadores_diferente_de_5` | algum time sem exatamente 5 jogadores | sim |
| `tipo_compra_ausente` | `team_a_type_buy` ou `team_b_type_buy` vazio | sim |
| `banco_diverge` | o banco da aba Economy ≠ a soma dos créditos dos jogadores (±50) | sim |
| `fora_de_faixa` | créditos fora de 0–45.000, loadout fora de 0–40.000 ou kills > 10 por time | sim |
| `round_duplicado` | (`game_id`, `round_number`) repetido | sim |
| `item_sem_preco` | arma ou armadura fora da tabela de preços | **não** (aviso: só afeta `tN_loadout_value`, que não é a variável principal de loadout) |

In [ ]:
def col(df, nome, padrao=np.nan):
    """Coluna do DataFrame ou uma série constante, se a coluna não existir (ex.: base sem aba Economy)."""
    return df[nome] if nome in df.columns else pd.Series(padrao, index=df.index)

REGRAS_ROUND = {
    # regra: (descrição, remove?)
    "vencedor_ausente": ("vencedor fora de {1, 2}", True),
    "vencedor_diverge": ("vencedor pelo placar diverge da faixa de rounds", True),
    "jogadores_diferente_de_5": ("time sem exatamente 5 jogadores", True),
    "tipo_compra_ausente": ("tipo de compra vazio", True),
    "banco_diverge": ("banco da aba Economy diverge da soma dos jogadores", True),
    "fora_de_faixa": ("créditos, loadout ou kills fora da faixa plausível", True),
    "round_duplicado": ("(game_id, round_number) repetido", True),
    "item_sem_preco": ("item fora da tabela de preços (aviso)", False),
}

def checar_rounds(r):
    c = pd.DataFrame(index=r.index)
    c["vencedor_ausente"] = ~col(r, "winner").isin([1, 2])
    c["vencedor_diverge"] = ~col(r, "winner_check_ok", False).fillna(False).astype(bool)
    c["jogadores_diferente_de_5"] = (col(r, "team_a_n_players", 0) != 5) | (col(r, "team_b_n_players", 0) != 5)
    c["tipo_compra_ausente"] = col(r, "team_a_type_buy").isna() | col(r, "team_b_type_buy").isna()
    c["banco_diverge"] = ~col(r, "banco_confere", False).fillna(False).astype(bool)
    fora = pd.Series(False, index=r.index)
    for t in ("a", "b"):
        cred = col(r, f"team_{t}_credits_left")
        load = col(r, f"team_{t}_loadout_vlr")
        kills = col(r, f"team_{t}_kills")
        # valor ausente não é "fora de faixa" (a ausência já é tratada por outras regras)
        fora |= (cred.notna() & ~cred.between(0, 45000)) | (load.notna() & ~load.between(0, 40000)) \
                | (kills.notna() & ~kills.between(0, 10))
    c["fora_de_faixa"] = fora
    c["round_duplicado"] = r.duplicated(["game_id", "round_number"], keep="first")
    c["item_sem_preco"] = (col(r, "team_a_n_itens_sem_preco", 0).fillna(0) + col(r, "team_b_n_itens_sem_preco", 0).fillna(0)) > 0
    return c

def checar_sequencia(r, mapas_ref):
    """Por mapa: sequência 1..n completa e placar final coerente."""
    out = []
    n_esperado = mapas_ref.drop_duplicates("game_id").set_index("game_id")["n_rounds"]
    for gid, g in r.sort_values("round_number").groupby("game_id"):
        nums = g["round_number"].drop_duplicates().tolist()
        n = len(nums)
        seq_ok = nums == list(range(1, n + 1)) and n_esperado.get(gid, n) == n
        s1, s2 = g["team_a_score_after"].iloc[-1], g["team_b_score_after"].iloc[-1]
        alto, dif = max(s1, s2), abs(s1 - s2)
        if n <= 24:
            placar_ok = (s1 + s2 == n) and alto == 13 and dif >= 1
        else:
            placar_ok = (s1 + s2 == n) and alto >= 13 and dif == 2
        out.append({"game_id": gid, "sequencia_incompleta": not seq_ok, "placar_inconsistente": not placar_ok})
    return pd.DataFrame(out, columns=["game_id", "sequencia_incompleta", "placar_inconsistente"])

In [ ]:
# ---------- aplicação das regras, em ordem (funil) ----------
M = mapas_jogados.copy()
R = rounds_raw.copy()
chk = checar_rounds(R)
remove_cols = [k for k, (_, rem) in REGRAS_ROUND.items() if rem]
R["round_invalido"] = chk[remove_cols].any(axis=1)

seq = checar_sequencia(R, M)
M = M.merge(seq, on="game_id", how="left")
M[["sequencia_incompleta", "placar_inconsistente"]] = M[["sequencia_incompleta", "placar_inconsistente"]].fillna(False)
mapas_com_round_invalido = set(R.loc[R["round_invalido"] & R["no_escopo"], "game_id"])

ROTULO_ESCOPO = {
    "showmatch": "showmatch (fora do escopo)",
    "open_qualifier": "open qualifier (fora do escopo)",
    "bo1": "série Bo1 (fora do escopo)",
    "evento_fora_da_lista": "evento fora de FILTRO_EVENTOS (fora do escopo)",
    "data_anterior": "partida antes de FILTRO_DATA_MIN (fora do escopo)",
}
passos = [(n, ROTULO_ESCOPO[n], (lambda m, n=n: m["motivo_escopo"].eq(n))) for n in ORDEM_ESCOPO] + [
    ("sem_replay",          "mapa sem 2D replay",                         lambda m: m["status"].eq("sem_replay")),
    ("nao_baixado",         "página não baixada ou não renderizada",      lambda m: m["status"].isin(["nao_baixado", "nao_renderizado"])),
    ("inconsistente_parser","inconsistência detectada pelo parser",       lambda m: m["status"].eq("inconsistente")),
    ("economia_invalida",   "aba Economy ausente ou divergente",          lambda m: m["status"].eq("ok") & ~col(m, "economia_status").eq("ok")),
    ("mapa_duplicado",      "game_id repetido",                           lambda m: m.duplicated("game_id", keep="first")),
    ("sequencia_incompleta","rounds faltando ou fora de ordem",           lambda m: m["sequencia_incompleta"]),
    ("placar_inconsistente","placar final não fecha",                     lambda m: m["placar_inconsistente"]),
]
if MAPA_INTEIRO:
    passos.append(("rounds_invalidos", "mapa com round inválido (MAPA_INTEIRO)", lambda m: m["game_id"].isin(mapas_com_round_invalido)))

restante = M.copy()
motivo = pd.Series(pd.NA, index=M.index, dtype="object")
funil_mapas = [("mapas jogados (antes da limpeza)", len(M), "")]
for regra, desc, cond in passos:
    sai = cond(restante).fillna(False).astype(bool)
    if regra in ORDEM_ESCOPO and not sai.any():
        continue                      # motivo de escopo desativado ou sem ocorrências: não polui o funil
    motivo.loc[restante.index[sai]] = regra
    funil_mapas.append((f"− {desc}", -int(sai.sum()), regra))
    restante = restante[~sai]
funil_mapas.append(("mapas válidos", len(restante), ""))
M["motivo_exclusao"] = motivo
mapas_validos = set(restante["game_id"])

funil_mapas = pd.DataFrame(funil_mapas, columns=["etapa", "mapas", "regra"])
display(funil_mapas)

In [ ]:
# ---------- funil dos rounds ----------
mapas_invalidos = set(M["game_id"]) - mapas_validos
mot_mapa = (M[M["game_id"].isin(mapas_invalidos)].dropna(subset=["motivo_exclusao"])
              .drop_duplicates("game_id").set_index("game_id")["motivo_exclusao"])
R["mapa_valido"] = R["game_id"].isin(mapas_validos)
R["motivo_mapa"] = R["game_id"].map(mot_mapa).astype("object")
R.loc[~R["game_id"].isin(set(M["game_id"])), "motivo_mapa"] = "mapa_ausente_na_tabela_mapas"

n_antes = len(R)
por_escopo = R["motivo_escopo"].notna()
por_mapa = ~por_escopo & R["motivo_mapa"].notna() & R["motivo_mapa"].ne("rounds_invalidos")
proprio = ~por_escopo & ~por_mapa & R["round_invalido"]
propagado = ~por_escopo & ~por_mapa & ~R["round_invalido"] & R["motivo_mapa"].eq("rounds_invalidos")

rounds_validos = R[~(por_escopo | por_mapa | proprio | propagado)].copy()
linhas = [("rounds coletados (antes da limpeza)", n_antes)]
for n in ORDEM_ESCOPO:
    k = int((R["motivo_escopo"] == n).sum())
    if k:
        linhas.append((f"− {ROTULO_ESCOPO[n]}", -k))
linhas += [
    ("− rounds de mapas inválidos (regras de mapa)", -int(por_mapa.sum())),
    ("− rounds inválidos (regras de round)", -int(proprio.sum())),
    ("− rounds válidos de mapas com round inválido (MAPA_INTEIRO)", -int(propagado.sum())),
    ("rounds válidos", len(rounds_validos)),
]
funil_rounds = pd.DataFrame(linhas, columns=["etapa", "rounds"])
assert funil_rounds["rounds"].iloc[:-1].sum() == len(rounds_validos)
display(funil_rounds)

# contagem por regra de round (não exclusiva: um round pode violar várias regras)
por_regra = pd.DataFrame({
    "descrição": {k: v[0] for k, v in REGRAS_ROUND.items()},
    "remove": {k: v[1] for k, v in REGRAS_ROUND.items()},
    "rounds que violam": chk[R["no_escopo"]].sum().astype(int),     # só rounds dentro do escopo
})
por_regra["% dos rounds"] = (100 * por_regra["rounds que violam"] / max(int(R["no_escopo"].sum()), 1)).round(2)
display(por_regra)

In [ ]:
resumo_limpeza = pd.DataFrame({
    "antes da limpeza": [len(M), n_antes],
    "fora do escopo": [int(M["motivo_escopo"].notna().sum()), int(por_escopo.sum())],
    "inválidos": [len(M) - int(M["motivo_escopo"].notna().sum()) - len(mapas_validos),
                  int((por_mapa | proprio | propagado).sum())],
    "válidos": [len(mapas_validos), len(rounds_validos)],
}, index=["mapas", "rounds"])
resumo_limpeza["% válidos (do total)"] = (100 * resumo_limpeza["válidos"] / resumo_limpeza["antes da limpeza"].clip(lower=1)).round(1)
no_esc = resumo_limpeza["antes da limpeza"] - resumo_limpeza["fora do escopo"]
resumo_limpeza["% válidos (do escopo)"] = (100 * resumo_limpeza["válidos"] / no_esc.clip(lower=1)).round(1)
assert (resumo_limpeza["fora do escopo"] + resumo_limpeza["inválidos"] + resumo_limpeza["válidos"]
        == resumo_limpeza["antes da limpeza"]).all()
display(resumo_limpeza)

fig, ax = plt.subplots(figsize=(8, 0.45 * len(funil_mapas) + 1))
cores = ["#7f7f7f"] + ["#c0504d"] * (len(funil_mapas) - 2) + ["#4f81bd"]
ax.barh(range(len(funil_mapas)), funil_mapas["mapas"].abs(), color=cores)
ax.set_yticks(range(len(funil_mapas)), funil_mapas["etapa"])
ax.invert_yaxis()   # de cima para baixo: antes da limpeza -> regras -> válidos
ax.set_xlabel("mapas")
ax.set_title("Funil de limpeza (mapas)")
for p in ax.patches:
    ax.text(p.get_width(), p.get_y() + p.get_height() / 2, f" {int(p.get_width())}", va="center", fontsize=8)
salvar_fig(fig, "funil_mapas")
plt.show()

**Como ler:** a tabela `resumo_limpeza` é o número para o texto do TCC: antes da limpeza, fora do
escopo (showmatch, open qualifier, Bo1 e filtros), inválidos e válidos, em mapas e em rounds. O funil mostra quanto cada regra removeu, aplicando as regras **em
ordem** (um mapa é contado só na primeira regra que o derruba). A tabela `por_regra` conta as
violações **sem exclusividade**, para mostrar a frequência de cada problema.

Se `sem_replay` dominar o funil, a perda vem da cobertura do site (eventos sem 2D replay), não de
erro na coleta. Vale citar isso na metodologia.

## 4. Análise descritiva da base válida

In [ ]:
V = rounds_validos.copy()
V["data"] = pd.to_datetime(V["date_utc"], errors="coerce")

por_evento = (V.groupby(["event_id", "event_name"])
                .agg(partidas=("match_id", "nunique"), mapas=("game_id", "nunique"), rounds=("round_number", "size"),
                     inicio=("data", "min"), fim=("data", "max"))
                .reset_index().sort_values("inicio"))
display(por_evento)

rpm = V.groupby("game_id").size()
print(f"rounds por mapa: média {rpm.mean():.1f} | mediana {rpm.median():.0f} | mín {rpm.min()} | máx {rpm.max()}")
print(f"mapas com prorrogação: {(rpm > 24).sum()} ({100 * (rpm > 24).mean():.1f}%)")
print(f"patches: {sorted(V['patch'].dropna().unique().tolist())}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
axs[0].hist(rpm, bins=range(13, int(rpm.max()) + 2), color="#4f81bd", edgecolor="white")
axs[0].set_title("Rounds por mapa"); axs[0].set_xlabel("rounds")
V["map_name"].value_counts().plot.barh(ax=axs[1], color="#4f81bd"); axs[1].set_title("Rounds por mapa (nome)")
V["win_type"].value_counts().plot.barh(ax=axs[2], color="#4f81bd"); axs[2].set_title("Tipo de vitória")
plt.tight_layout(); salvar_fig(fig, "descritiva_mapas"); plt.show()

### 4.1 Formato por time e variáveis processadas

Cada round vira **duas linhas**, uma por time (perspectiva do `team_a` e do `team_b`). Todas as
variáveis acumuladas são calculadas com o histórico **até o round anterior** (`shift(1)`), ou
seja, só com o que é conhecido quando o round começa:

| Variável | Definição |
|---|---|
| `win_count`, `loss_count` | vitórias e derrotas do time no mapa antes do round (do placar oficial) |
| `score_diff` | `win_count − loss_count` (placar antes do round, do ponto de vista do time) |
| `historico` | rounds já disputados no mapa (= `round_number − 1`) |
| `streak` | sequência atual antes do round: +k = k vitórias seguidas, −k = k derrotas seguidas |
| `loss_streak` | derrotas seguidas antes do round, zerada na troca de lado (bônus de derrota) |
| `prev_win`, `prev_win_type` | resultado e tipo de vitória do round anterior |
| `kills_prev`, `kills_diff_prev`, `kills_diff_media3` | kills do round anterior e tendência (média móvel de 3 rounds) |
| `type_buy`, `opp_type_buy`, `loadout`, `opp_loadout`, `loadout_diff`, `credits`, `credits_diff` | economia do round atual (conhecida após a fase de compra) |
| `win` | **alvo**: 1 se o time venceu o round |

Atenção a duas colunas: `kills` e `opp_kills` são do round **atual** e só podem entrar no modelo
defasadas (`kills_prev`). Elas ficam na tabela para análise.

In [ ]:
def para_formato_time(r):
    base_cols = ["event_id", "event_name", "match_id", "game_id", "map_name", "patch", "date_utc",
                 "round_number", "half", "is_pistol_round", "is_overtime", "win_type"]
    partes = []
    for t, o in (("a", "b"), ("b", "a")):
        d = r[base_cols].copy()
        d["perspectiva"] = t
        d["team"], d["opponent"] = r[f"team_{t}"], r[f"team_{o}"]
        d["side"] = r[f"team_{t}_side"]
        d["type_buy"], d["opp_type_buy"] = r[f"team_{t}_type_buy"], r[f"team_{o}_type_buy"]
        d["loadout"], d["opp_loadout"] = r[f"team_{t}_loadout_vlr"], r[f"team_{o}_loadout_vlr"]
        d["loadout_arma_armadura"] = r[f"team_{t}_loadout_value"]
        d["credits"], d["opp_credits"] = r[f"team_{t}_credits_left"], r[f"team_{o}_credits_left"]
        d["kills"], d["opp_kills"] = r[f"team_{t}_kills"], r[f"team_{o}_kills"]
        d["win"] = (r["winner"] == (1 if t == "a" else 2)).astype(int)
        d["_score_after"] = r[f"team_{t}_score_after"]
        partes.append(d)
    L = pd.concat(partes, ignore_index=True).sort_values(["game_id", "perspectiva", "round_number"])

    g = L.groupby(["game_id", "perspectiva"], sort=False)
    L["historico"] = L["round_number"] - 1
    # placar antes do round, a partir do placar oficial (correto mesmo se faltar algum round)
    L["win_count"] = L["_score_after"] - L["win"]
    L["loss_count"] = L["historico"] - L["win_count"]
    L["score_diff"] = L["win_count"] - L["loss_count"]
    L["prev_win"] = g["win"].shift(1)
    L["prev_win_type"] = g["win_type"].shift(1)
    L["kills_prev"] = g["kills"].shift(1)
    L["kills_diff_prev"] = (L["kills"] - L["opp_kills"]).groupby([L["game_id"], L["perspectiva"]]).shift(1)
    L["kills_diff_media3"] = (L.groupby(["game_id", "perspectiva"])["kills_diff_prev"]
                                .transform(lambda s: s.rolling(3, min_periods=1).mean()))

    # streak com sinal, antes do round
    def streak_antes(w):
        out, cur = [], 0
        for x in w:
            out.append(cur)
            cur = (cur + 1 if cur > 0 else 1) if x == 1 else (cur - 1 if cur < 0 else -1)
        return out
    L["streak"] = g["win"].transform(lambda s: pd.Series(streak_antes(s.tolist()), index=s.index))

    # derrotas seguidas antes do round, zerando na troca de lado (round 13) e a cada round de prorrogação
    bloco = np.where(L["round_number"] <= 12, 0, np.where(L["round_number"] <= 24, 1, L["round_number"]))
    L["_bloco"] = bloco
    def loss_antes(w):
        out, cur = [], 0
        for x in w:
            out.append(cur)
            cur = 0 if x == 1 else cur + 1
        return out
    L["loss_streak"] = (L.groupby(["game_id", "perspectiva", "_bloco"])["win"]
                          .transform(lambda s: pd.Series(loss_antes(s.tolist()), index=s.index)))
    L = L.drop(columns="_bloco")

    # se faltar algum round no mapa (só acontece com MAPA_INTEIRO = False), as variáveis de
    # sequência deixam de ser confiáveis a partir da lacuna: ficam vazias
    lacuna = (L.groupby(["game_id", "perspectiva"])["round_number"].diff().fillna(1) != 1)
    apos_lacuna = lacuna.groupby([L["game_id"], L["perspectiva"]]).cumsum() > 0
    L["apos_lacuna"] = apos_lacuna
    L.loc[apos_lacuna, ["prev_win", "prev_win_type", "kills_prev", "kills_diff_prev", "kills_diff_media3",
                        "streak", "loss_streak"]] = np.nan
    L = L.drop(columns="_score_after")

    L["loadout_diff"] = L["loadout"] - L["opp_loadout"]
    L["credits_diff"] = L["credits"] - L["opp_credits"]
    L["outcome"] = np.where(L["win"] == 1, "Win", "Loss")
    return L.reset_index(drop=True)

L = para_formato_time(V)
print(L.shape)
L.head(3)

In [ ]:
# conferências de vazamento e coerência das variáveis processadas
assert (L.loc[L["round_number"] == 1, ["win_count", "loss_count", "score_diff", "streak"]] == 0).all().all()
assert L.loc[L["round_number"] == 1, "prev_win"].isna().all()
assert (L["win_count"] + L["loss_count"] == L["historico"]).all()
completos = L.groupby("game_id")["apos_lacuna"].transform("max").eq(False)
Lc = L[completos].sort_values(["game_id", "perspectiva", "round_number"])
cum = Lc.groupby(["game_id", "perspectiva"])["win"].cumsum() - Lc["win"]
assert (cum == Lc["win_count"]).all(), "win_count diverge da soma das vitórias anteriores"
print(f"mapas com lacuna (variáveis de sequência vazias após a lacuna): {L.loc[~completos, 'game_id'].nunique()}")
print("ok: variáveis acumuladas usam só rounds anteriores e batem com o placar final")

In [ ]:
num = ["kills", "loadout", "loadout_arma_armadura", "credits", "loadout_diff", "credits_diff",
       "win_count", "loss_count", "score_diff", "streak", "loss_streak"]
desc = L[num].describe(percentiles=[.05, .25, .5, .75, .95]).T
desc["faltantes"] = L[num].isna().sum()
display(desc.round(1))

fig, axs = plt.subplots(2, 3, figsize=(14, 6.5))
for ax, c in zip(axs.ravel(), ["kills", "loadout", "credits", "loadout_diff", "score_diff", "streak"]):
    s = L[c].dropna()
    ax.hist(s, bins=min(40, max(5, s.nunique())), color="#4f81bd", edgecolor="white")
    ax.set_title(c)
plt.tight_layout(); salvar_fig(fig, "distribuicoes"); plt.show()

dist_buy = (L.groupby(["type_buy"]).size().rename("linhas").to_frame()
              .assign(pct=lambda d: (100 * d["linhas"] / d["linhas"].sum()).round(1)))
display(dist_buy)

## 5. Análise estatística

**Unidade de análise.** Nos testes, cada round entra **uma vez**, na perspectiva do `team_a`
(`perspectiva == "a"`). Assim, as duas linhas do mesmo round, que são complementares, não contam em
dobro. A exceção são as análises por lado, que usam a perspectiva do atacante.

**Dependência.** Rounds do mesmo mapa não são independentes. Os testes clássicos (qui-quadrado,
binomial) assumem independência e tendem a superestimar a significância. Por isso, as taxas vêm
com **intervalo de confiança de 95% por bootstrap de mapas**, que reamostra mapas inteiros e
mantém a dependência dentro do mapa. Nas conclusões, use preferencialmente esses intervalos.

In [ ]:
def boot_taxa(df, y="win", cluster="game_id", n_boot=N_BOOT):
    """Taxa média de y com IC 95% por bootstrap de clusters (mapas)."""
    if len(df) == 0:
        return np.nan, np.nan, np.nan
    agg = df.groupby(cluster)[y].agg(["sum", "count"])
    s, n = agg["sum"].to_numpy(float), agg["count"].to_numpy(float)
    if len(agg) < 2:
        return s.sum() / n.sum(), np.nan, np.nan
    idx = rng.integers(0, len(agg), size=(n_boot, len(agg)))
    taxas = s[idx].sum(1) / n[idx].sum(1)
    return s.sum() / n.sum(), *np.percentile(taxas, [2.5, 97.5])

def tabela_taxas(df, grupo, y="win", min_n=1):
    linhas = []
    for k, g in df.groupby(grupo, dropna=False):
        if len(g) < min_n:
            continue
        p, lo, hi = boot_taxa(g, y)
        linhas.append({**dict(zip(grupo if isinstance(grupo, list) else [grupo],
                                  k if isinstance(k, tuple) else (k,))),
                       "rounds": len(g), "mapas": g["game_id"].nunique(),
                       "taxa_vitoria": round(p, 3), "ic95_inf": round(lo, 3), "ic95_sup": round(hi, 3)})
    return pd.DataFrame(linhas)

POR_ROUND = L[L["perspectiva"] == "a"].copy()
ATK = L[L["side"] == "atk"].copy()
print(f"rounds (uma linha por round): {len(POR_ROUND)} | mapas: {POR_ROUND['game_id'].nunique()}")

### 5.1 Lado (ataque × defesa) por mapa

H0: a taxa de vitória do ataque é 50%. Teste binomial, com IC por bootstrap de mapas.

In [ ]:
lado = tabela_taxas(ATK, "map_name").rename(columns={"taxa_vitoria": "vitoria_ataque"})
lado["p_binomial"] = [stats.binomtest(int(round(r.vitoria_ataque * r.rounds)), int(r.rounds), 0.5).pvalue
                      for r in lado.itertuples()]
geral = boot_taxa(ATK)
print(f"vitória do ataque, geral: {geral[0]:.3f} (IC95 {geral[1]:.3f}–{geral[2]:.3f}), "
      f"p binomial = {stats.binomtest(int(ATK['win'].sum()), len(ATK), 0.5).pvalue:.4f}")
display(lado.sort_values("rounds", ascending=False))

### 5.2 Tipo de compra: taxa de vitória por confronto

É a tabela equivalente à Tabela 1 de Xenopoulos et al. (2021) para o CS:GO: taxa de vitória do
time de acordo com a própria compra (linhas) e a do adversário (colunas). Ela usa as duas
perspectivas (é uma tabela descritiva, e a matriz é complementar). O teste de independência
qui-quadrado entre o tipo de compra e a vitória usa uma linha por round.

In [ ]:
ordem = [b for b in ["Pistol", "Eco", "Half-buy", "Fullbuy"] if b in set(L["type_buy"].dropna())]
taxa = L.pivot_table(index="type_buy", columns="opp_type_buy", values="win", aggfunc="mean").reindex(index=ordem, columns=ordem)
cont = L.pivot_table(index="type_buy", columns="opp_type_buy", values="win", aggfunc="size").reindex(index=ordem, columns=ordem)
print("Taxa de vitória (linha = compra do time; coluna = compra do adversário)")
display(taxa.round(3))
print("Número de rounds-time")
display(cont.fillna(0).astype(int))

ct = pd.crosstab(POR_ROUND["type_buy"], POR_ROUND["win"])
if ct.shape[0] > 1 and ct.shape[1] > 1:
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n = ct.to_numpy().sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    print(f"qui-quadrado tipo de compra × vitória: χ²={chi2:.1f}, gl={dof}, p={p:.2e}, V de Cramér={v:.3f}")
else:
    print("amostra sem variação suficiente para o qui-quadrado")

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(taxa.to_numpy(dtype=float), cmap="RdBu", vmin=0, vmax=1)
ax.set_xticks(range(len(ordem)), ordem); ax.set_yticks(range(len(ordem)), ordem)
ax.set_xlabel("compra do adversário"); ax.set_ylabel("compra do time")
for i in range(len(ordem)):
    for j in range(len(ordem)):
        val = taxa.to_numpy(dtype=float)[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="taxa de vitória"); ax.set_title("Vitória por confronto de compras")
salvar_fig(fig, "matriz_tipo_compra"); plt.show()

### 5.3 Diferença de loadout e vitória

Correlação ponto-bisserial e AUC da `loadout_diff` como único preditor (rounds de pistol excluídos, porque o loadout é igual por regra).

In [ ]:
X = POR_ROUND[~POR_ROUND["is_pistol_round"].astype(bool)].dropna(subset=["loadout_diff"])
if X["win"].nunique() == 2:
    r_pb, p_pb = stats.pointbiserialr(X["win"], X["loadout_diff"])
    print(f"ponto-bisserial: r={r_pb:.3f}, p={p_pb:.2e} | AUC = {roc_auc_score(X['win'], X['loadout_diff']):.3f}")
faixas = pd.cut(X["loadout_diff"], bins=np.arange(-30000, 30001, 5000))
curva = tabela_taxas(X.assign(faixa=faixas.astype(str)), "faixa")
curva["_ord"] = curva["faixa"].str.extract(r"\((-?\d+)")[0].astype(float)
curva = curva.sort_values("_ord").drop(columns="_ord")
display(curva)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.errorbar(range(len(curva)), curva["taxa_vitoria"],
            yerr=[curva["taxa_vitoria"] - curva["ic95_inf"], curva["ic95_sup"] - curva["taxa_vitoria"]],
            fmt="o-", color="#4f81bd", capsize=3)
ax.set_xticks(range(len(curva)), curva["faixa"], rotation=45, ha="right", fontsize=7)
ax.axhline(0.5, color="grey", lw=0.8, ls="--"); ax.set_ylim(0, 1)
ax.set_ylabel("taxa de vitória"); ax.set_title("Vitória por diferença de loadout (IC95 por bootstrap de mapas)")
salvar_fig(fig, "loadout_diff"); plt.show()

### 5.4 Momentum (H3)

H3 diz que o momentum pesa mais que o estado econômico do round. O problema é que os dois se
confundem: quem vence ganha mais créditos e tende a comprar melhor no round seguinte. Por isso o
efeito do round anterior é medido de duas formas:

1. **Na base inteira:** P(vitória | venceu o anterior) × P(vitória | perdeu o anterior).
2. **Controlando a economia:** a mesma comparação só em rounds com **Fullbuy × Fullbuy**, em que
   os dois times compraram completo. Se o efeito persistir aqui, há indício de momentum além da
   economia.

Isso é evidência descritiva e preliminar para H3. O teste propriamente dito vem dos modelos
(importância das variáveis e ablação).

In [ ]:
def comparar_prev(df, rotulo):
    d = df.dropna(subset=["prev_win"])
    ct = pd.crosstab(d["prev_win"], d["win"])
    linha = {"subconjunto": rotulo, "rounds": len(d)}
    for v, nome in ((1, "venceu_anterior"), (0, "perdeu_anterior")):
        p, lo, hi = boot_taxa(d[d["prev_win"] == v])
        linha[f"P(vit|{nome})"] = round(p, 3)
        linha[f"IC95 {nome}"] = f"{lo:.3f}–{hi:.3f}" if not np.isnan(lo) else "—"
    if ct.shape == (2, 2):
        chi2, p, _, _ = stats.chi2_contingency(ct)
        a, b, c, e = ct.loc[1, 1], ct.loc[1, 0], ct.loc[0, 1], ct.loc[0, 0]
        linha["odds_ratio"] = round((a * e) / max(b * c, 1e-9), 2)
        linha["p_qui2"] = p
    return linha

ff = POR_ROUND[(POR_ROUND["type_buy"] == "Fullbuy") & (POR_ROUND["opp_type_buy"] == "Fullbuy")]
display(pd.DataFrame([comparar_prev(POR_ROUND, "todos os rounds"), comparar_prev(ff, "Fullbuy × Fullbuy")]))

def faixa_streak(v):
    if pd.isna(v):
        return pd.NA
    if v == 0:
        return "0 (1º round)"
    return f"{int(v):+d}" + ("+" if abs(v) == 3 else "")
POR_ROUND["streak_faixa"] = POR_ROUND["streak"].clip(-3, 3).map(faixa_streak)
ordem_s = ["-3+", "-2", "-1", "0 (1º round)", "+1", "+2", "+3+"]
st = tabela_taxas(POR_ROUND.dropna(subset=["streak_faixa"]), "streak_faixa").set_index("streak_faixa").reindex([o for o in ordem_s if o in set(POR_ROUND["streak_faixa"])])
display(st)

### 5.5 Poder preditivo de cada variável isolada (AUC)

A AUC de uma variável sozinha mede quanto ela ordena vencedores acima de perdedores: 0,5 é
aleatório. Valores abaixo de 0,5 indicam relação inversa (a "força" é |AUC − 0,5|). Os IC vêm de
bootstrap de mapas. É uma comparação **univariada**: serve como referência para H3, sem substituir
a análise de importância nos modelos.

In [ ]:
def auc_boot(df, x, y="win", n_boot=N_BOOT):
    d = df.dropna(subset=[x])
    if d[y].nunique() < 2:
        return np.nan, np.nan, np.nan
    base = roc_auc_score(d[y], d[x])
    mapas = d["game_id"].unique()
    grupos = {g: idx for g, idx in d.groupby("game_id").indices.items()}
    yv, xv = d[y].to_numpy(), d[x].to_numpy()
    vals = []
    for _ in range(min(n_boot, 300)):
        sel = np.concatenate([grupos[g] for g in rng.choice(mapas, len(mapas))])
        if len(np.unique(yv[sel])) == 2:
            vals.append(roc_auc_score(yv[sel], xv[sel]))
    lo, hi = (np.percentile(vals, [2.5, 97.5]) if len(vals) > 10 else (np.nan, np.nan))
    return base, lo, hi

Xnp = POR_ROUND[~POR_ROUND["is_pistol_round"].astype(bool)]
variaveis = {
    "loadout_diff": "economia", "credits_diff": "economia", "loadout": "economia",
    "prev_win": "momentum", "streak": "momentum", "score_diff": "momentum (placar)",
    "kills_diff_prev": "momentum (kills)", "kills_diff_media3": "momentum (kills)",
}
aucs = []
for v, grupo in variaveis.items():
    a, lo, hi = auc_boot(Xnp, v)
    aucs.append({"variável": v, "grupo": grupo, "AUC": round(a, 3), "IC95 inf": round(lo, 3), "IC95 sup": round(hi, 3),
                 "força |AUC−0,5|": round(abs(a - 0.5), 3)})
aucs = pd.DataFrame(aucs).sort_values("força |AUC−0,5|", ascending=False)
display(aucs)

### 5.6 Conversão após o pistol

Probabilidade de vencer os rounds 2 e 3 (e 14 e 15) dado o resultado do pistol da metade.

In [ ]:
pist = L[L["round_number"].isin([1, 13])][["game_id", "perspectiva", "round_number", "win"]].rename(columns={"win": "venceu_pistol"})
pist["metade"] = np.where(pist["round_number"] == 1, 1, 2)
seg = L[L["round_number"].isin([2, 3, 14, 15])].copy()
seg["metade"] = np.where(seg["round_number"] <= 12, 1, 2)
seg["rounds_apos_pistol"] = np.where(seg["round_number"].isin([2, 14]), 1, 2)
seg = seg.merge(pist.drop(columns="round_number"), on=["game_id", "perspectiva", "metade"])
conv = tabela_taxas(seg[seg["perspectiva"] == "a"], ["rounds_apos_pistol", "venceu_pistol"])
display(conv)

### 5.7 Tamanho das sequências (H1 e H2)

H1 e H2 dependem de quanto histórico existe antes de cada round. A tabela mostra quantos rounds
têm pelo menos *k* rounds anteriores no mapa. É o tamanho de amostra disponível para comparar o
LSTM com os modelos de snapshot em cada faixa de histórico (o "cold start" de H2).

In [ ]:
ks = [0, 1, 3, 5, 8, 12, 18, 24]
hist = pd.DataFrame({"histórico mínimo (k)": ks,
                     "rounds com histórico ≥ k": [int((POR_ROUND["historico"] >= k).sum()) for k in ks]})
hist["% dos rounds"] = (100 * hist["rounds com histórico ≥ k"] / max(len(POR_ROUND), 1)).round(1)
display(hist)

### 5.8 Correlação entre as variáveis (Spearman)

Serve para identificar redundância antes da modelagem. Por construção, `score_diff = 2·win_count − historico`
e `win_count + loss_count = historico`: essas variáveis são **linearmente dependentes**. Num modelo
linear, use só duas delas (por exemplo, `score_diff` e `historico`). Para árvores e LSTM, a
redundância não impede o treino, mas dilui a importância entre elas.

In [ ]:
cols_corr = ["win", "loadout_diff", "credits_diff", "loadout", "win_count", "loss_count", "score_diff",
             "historico", "streak", "loss_streak", "prev_win", "kills_diff_prev"]
corr = POR_ROUND[cols_corr].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(corr.to_numpy(), cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_corr)), cols_corr, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(cols_corr)), cols_corr, fontsize=8)
for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        v = corr.to_numpy()[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax); ax.set_title("Correlação de Spearman (uma linha por round)")
salvar_fig(fig, "correlacao"); plt.show()

## 6. Salvar as saídas da transformação

In [ ]:
if SALVAR:
    def gravar(df, nome):
        df = df.copy()
        for c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].astype("string")
        df.to_parquet(PASTA_SAIDA / f"{nome}.parquet", index=False, compression="zstd")
        print(f"{nome:28s} {len(df):8d} linhas -> {PASTA_SAIDA / (nome + '.parquet')}")

    gravar(rounds_validos.drop(columns=["no_escopo", "motivo_escopo", "round_invalido", "mapa_valido", "motivo_mapa"]), "rounds_validos")
    gravar(L, "rounds_time")                      # formato por time, com as variáveis processadas
    gravar(M[["event_id", "match_id", "game_id", "map_name", "status", "motivo_escopo", "motivo_exclusao"]], "mapas_limpeza")
    gravar(partidas[["event_id", "event_name", "match_id", "event_series", "best_of", "tipo_showmatch",
                     "tipo_open_qualifier", "tipo_bo1", "bo1_inferido", "motivo_escopo"]], "partidas_escopo")
    gravar(funil_mapas, "funil_mapas")
    gravar(funil_rounds, "funil_rounds")
    gravar(por_regra.reset_index(names="regra"), "regras_round")
    gravar(resumo_limpeza.reset_index(names="nivel"), "resumo_limpeza")
    print("figuras:", sorted(p.name for p in PASTA_FIG.glob("*.png")))

## 7. Resumo

In [ ]:
print("=" * 70)
print("RESUMO DA LIMPEZA")
print("=" * 70)
for nivel in ("mapas", "rounds"):
    r = resumo_limpeza.loc[nivel]
    print(f"{nivel:7s}: antes {int(r['antes da limpeza']):6d} | fora do escopo {int(r['fora do escopo']):6d} | "
          f"inválidos {int(r['inválidos']):6d} | válidos {int(r['válidos']):6d} ({r['% válidos (do total)']:.1f}% do total)")
print(f"\nbase final (formato por time): {len(L)} linhas = {len(POR_ROUND)} rounds × 2 perspectivas, "
      f"{POR_ROUND['game_id'].nunique()} mapas, {POR_ROUND['match_id'].nunique()} partidas, {POR_ROUND['event_id'].nunique()} eventos")
print(f"política: MAPA_INTEIRO={MAPA_INTEIRO} | excluir showmatch={EXCLUIR_SHOWMATCH}, open qualifier="
      f"{EXCLUIR_OPEN_QUALIFIER}, Bo1={EXCLUIR_BO1} | eventos={FILTRO_EVENTOS}, data ≥ {FILTRO_DATA_MIN}")